# LeetCode 655 — Print Binary Tree bằng Python

Link đề: https://leetcode.com/problems/print-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def printTree(self, root: Optional[TreeNode]) -> List[List[str]]:
        ...
```

Ghi chú: kết quả là **bảng (ma trận) các chuỗi** với `""` ở ô trống, nên test so sánh **từng ô** theo đúng quy tắc của đề. Notebook tự dựng `TreeNode` và hàm `build_tree` dựng cây từ list kiểu LeetCode (ví dụ `[1,2,3,None,4]`). Các ví dụ số 3, 4 và các cây tự kiểm do notebook tự đặt, đáp án đã kiểm chứng bằng code; nếu cần hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho một cây nhị phân, in nó vào **bảng** `m × n` (`m = height + 1` hàng, `n = 2^(height+1) − 1` cột); gốc nằm **giữa hàng trên cùng**; con của nút ở `(r, c)` nằm ở hàng `r+1`, cột `c ∓ 2^(height−r−1)`; ô trống là chuỗi rỗng `""`.
2. Nắm các công cụ Python: `TreeNode`, dựng cây từ list, **chiều cao cây** bằng đệ quy, luỹ thừa của 2 (`2 ** k`, `1 << k`), tạo ma trận 2 chiều **đúng cách** (`[[""] * n for _ in range(m)]`), `str()` cho giá trị âm và `0`, hàng đợi `deque` cho BFS.
3. Đi từ **ghép các bảng con** (đệ quy chia đôi) → **hai lượt DFS: đo chiều cao rồi đặt từng nút bằng công thức độ lệch** (cách chuẩn) → **BFS lặp, không đệ quy** → **công thức đóng: cột theo chỉ số đống (heap index)**.
4. Hiểu mấu chốt: biết **chiều cao `h`** trước thì biết kích thước bảng, và **khoảng cách giữa con và cha giảm một nửa theo từng tầng** (`2^(h−r−1)`), nên mỗi nút có vị trí xác định mà **không cần biết nút khác**.
5. Tránh các lỗi kinh điển: `[[""] * n] * m` (các hàng dùng chung một list), đếm chiều cao bằng số nút thay vì số cạnh, lệch `2 ** (h − r)` thay vì `2 ** (h − r − 1)` (chỉ số âm **âm thầm** ghi sai chỗ), quên `str()`, và bỏ sót nút có giá trị `0`.

## 2. Đề bài & ví dụ

Cho nút gốc `root` của một cây nhị phân. Hãy dựng một ma trận chuỗi `res` **đánh chỉ số từ 0**, kích thước `m × n`, biểu diễn **bố cục có định dạng** của cây theo các quy tắc:

- **Chiều cao** của cây là `height` (số **cạnh** trên đường dài nhất từ gốc xuống lá). Số hàng `m = height + 1`.
- Số cột `n = 2^(height+1) − 1`.
- Đặt **gốc** ở **giữa hàng trên cùng**: `res[0][(n − 1) / 2]`.
- Với mỗi nút đã đặt tại `res[r][c]`: đặt **con trái** tại `res[r+1][c − 2^(height−r−1)]` và **con phải** tại `res[r+1][c + 2^(height−r−1)]`.
- Lặp đến khi đặt hết các nút.
- Mọi ô còn trống chứa chuỗi rỗng `""`.

Trả về ma trận `res`.

**Ví dụ 1**

- Input: `root = [1,2]`
- Output:

```
[["", "1", ""],
 ["2", "", ""]]
```

**Ví dụ 2**

- Input: `root = [1,2,3,null,4]`
- Output:

```
[["", "", "", "1", "", "", ""],
 ["", "2", "", "", "", "3", ""],
 ["", "", "4", "", "", "", ""]]
```

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `root = [5]`
- Output: `[["5"]]` (`height = 0`, bảng `1 × 1`)

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `root = [1,null,2]`
- Output: `[["", "1", ""], ["", "", "2"]]`

**Ràng buộc:**

- Số nút của cây nằm trong khoảng `[1, 2^10]`.
- `-99 <= Node.val <= 99`
- Độ sâu của cây nằm trong khoảng `[1, 10]`.

Chú ý: kích thước bảng chỉ phụ thuộc **chiều cao**, không phụ thuộc số nút: một cây lệch chỉ 11 nút vẫn cho bảng `11 × 2047` (nhiều ô trống).

In [1]:
from typing import Optional, List
from collections import deque
import random
import sys
import time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def random_tree_list(rng, n, lo, hi):
    """Sinh cây ngẫu nhiên n nút (n >= 1) với giá trị trong [lo, hi], trả về list kiểu LeetCode."""
    root = TreeNode(rng.randint(lo, hi))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        k = rng.randrange(len(slots))
        slots[k], slots[-1] = slots[-1], slots[k]
        parent, side = slots.pop()
        child = TreeNode(rng.randint(lo, hi))
        setattr(parent, side, child)
        slots.append((child, "left"))
        slots.append((child, "right"))
    return tree_to_list(root)


examples = [
    ([1, 2], [["", "1", ""],
              ["2", "", ""]]),
    ([1, 2, 3, None, 4], [["", "", "", "1", "", "", ""],
                          ["", "2", "", "", "", "3", ""],
                          ["", "", "4", "", "", "", ""]]),
    ([5], [["5"]]),                                         # ví dụ tự đặt
    ([1, None, 2], [["", "1", ""], ["", "", "2"]]),         # ví dụ tự đặt
]
for vals, expected in examples:
    assert tree_to_list(build_tree(vals)) == vals
    print(vals, "-> bảng", len(expected), "x", len(expected[0]))

[1, 2] -> bảng 2 x 3
[1, 2, 3, None, 4] -> bảng 3 x 7
[5] -> bảng 1 x 1
[1, None, 2] -> bảng 2 x 3


## 3. Kiến thức nền

### 3.1. Nút cây và dựng cây từ list kiểu LeetCode

Mỗi nút `TreeNode` có `val`, `left`, `right`. Đề cho cây dạng list `[1,2,3,None,4]` theo **thứ tự từng tầng, từ trái sang phải**, `None` là chỗ trống; `build_tree` (ở cell trên) dựng cây, `tree_to_list` làm ngược lại. `show_levels` in từng tầng:

In [2]:
def show_levels(root):
    level, d = [root], 0
    while any(x is not None for x in level):
        print(f"tầng {d}:", [x.val if x else None for x in level])
        level = [c for x in level if x for c in (x.left, x.right)]
        d += 1

t = build_tree([1, 2, 3, None, 4])
show_levels(t)
print(tree_to_list(t))

tầng 0: [1]
tầng 1: [2, 3]
tầng 2: [None, 4, None, None]
[1, 2, 3, None, 4]


### 3.2. Chiều cao của cây: đếm **cạnh**, không đếm nút

**Chiều cao** (`height`) theo đề là số **cạnh** trên đường dài nhất từ gốc xuống lá. Cây một nút có `height = 0`. Công thức đệ quy: `height(None) = −1` (quy ước "cây rỗng cao −1") và `height(node) = 1 + max(height(left), height(right))`. Với quy ước này lá có `1 + max(−1, −1) = 0`.

In [3]:
def tree_height(root):
    if root is None:
        return -1
    return 1 + max(tree_height(root.left), tree_height(root.right))

print(tree_height(None), tree_height(build_tree([5])))             # -1 0
print(tree_height(build_tree([1, 2])))                              # 1
print(tree_height(build_tree([1, 2, 3, None, 4])))                  # 2
print(tree_height(build_tree([1, None, 2, None, 3, None, 4])))      # 3 (cây lệch)

-1 0
1
2
3


### 3.3. Luỹ thừa của 2 và công thức kích thước bảng

`2 ** k` là `2` mũ `k`. Cách khác: `1 << k` (dịch bit trái `k` lần, nhanh hơn và luôn cho số **nguyên**). Với `height = h`: `m = h + 1` hàng, `n = 2 ** (h + 1) − 1` cột, gốc ở cột `(n − 1) // 2` (vì `n` lẻ nên `//` cho số nguyên; ghi chú: chia `/` sẽ ra `float`, không dùng làm chỉ số được).

Chú ý: `2 ** (−1)` là `0.5` (số thực), **không** gây lỗi nhưng cho chỉ số sai kiểu; `1 << (−1)` thì báo `ValueError`. Vì thế phải đảm bảo mũ không âm khi tính độ lệch.

In [4]:
for h in range(0, 5):
    m, n = h + 1, 2 ** (h + 1) - 1
    print(f"h={h}: bảng {m} x {n}, gốc ở cột {(n - 1) // 2} (= 2^h - 1 = {2 ** h - 1}), 1<<h = {1 << h}")
print(2 ** (-1), type(7 / 2), 7 // 2)
try:
    1 << -1
except ValueError as e:
    print("ValueError:", e)

h=0: bảng 1 x 1, gốc ở cột 0 (= 2^h - 1 = 0), 1<<h = 1
h=1: bảng 2 x 3, gốc ở cột 1 (= 2^h - 1 = 1), 1<<h = 2
h=2: bảng 3 x 7, gốc ở cột 3 (= 2^h - 1 = 3), 1<<h = 4
h=3: bảng 4 x 15, gốc ở cột 7 (= 2^h - 1 = 7), 1<<h = 8
h=4: bảng 5 x 31, gốc ở cột 15 (= 2^h - 1 = 15), 1<<h = 16
0.5 <class 'float'> 3
ValueError: negative shift count


### 3.4. Tạo ma trận 2 chiều **đúng cách**

Ma trận `m × n` toàn `""`:

```python
res = [[""] * n for _ in range(m)]          # ĐÚNG: mỗi hàng là một list RIÊNG
```

Cách sai thường gặp: `[[""] * n] * m` — nó tạo **một** list hàng rồi lặp **tham chiếu** `m` lần, nên sửa một hàng là sửa **mọi** hàng.

In [5]:
bad = [[""] * 3] * 2
bad[0][1] = "x"
print("sai :", bad)                   # cả hai hàng đều đổi
good = [[""] * 3 for _ in range(2)]
good[0][1] = "x"
print("đúng:", good)
print(bad[0] is bad[1], good[0] is good[1])

sai : [['', 'x', ''], ['', 'x', '']]
đúng: [['', 'x', ''], ['', '', '']]
True False


### 3.5. Hàng đợi `deque`, tuple và hàm lồng nhau

- `deque` là hàng đợi hai đầu: `q.append(x)` thêm cuối, `q.popleft()` lấy đầu, `len(q)` đếm phần tử. Dùng cho **duyệt theo tầng (BFS)**.
- Ta có thể đưa **bộ ba** `(node, r, c)` vào hàng đợi rồi tách bằng `node, r, c = q.popleft()` (unpacking).
- Hàm lồng nhau (`def dfs` bên trong hàm chính) thấy được biến của hàm ngoài (như `res`, `h`) mà **không cần** `nonlocal`, miễn là ta chỉ **đọc** biến hoặc **sửa nội dung** (như `res[r][c] = ...`) chứ không **gán lại** tên biến.

In [6]:
q = deque()
q.append(("gốc", 0, 3))
q.append(("trái", 1, 1))
node, r, c = q.popleft()
print(node, r, c, "| còn", len(q))

def outer():
    res = [0, 0]
    def inner(i):
        res[i] += 1              # sửa nội dung: không cần nonlocal
    inner(0); inner(1); inner(1)
    return res
print(outer())

gốc 0 3 | còn 1
[1, 2]


### 3.6. Hiển thị bảng cho dễ nhìn và kiểm tra bố cục

Bảng toàn chuỗi khó đọc khi in thẳng. Hàm `show` in mỗi hàng, ô trống thay bằng dấu `.`, mỗi ô rộng đủ cho chuỗi dài nhất (giá trị có thể là `-99`). `check_layout` là **hàm kiểm tra theo đúng quy tắc của đề**: nó đi dọc cây, tính vị trí `(r, c)` từng nút bằng công thức đề, rồi kiểm tra (1) kích thước bảng, (2) mỗi nút nằm đúng ô với đúng chuỗi, (3) mọi ô khác là `""`. Ta dùng nó như "trọng tài" độc lập với các cách dựng.

In [7]:
def show(res):
    w = max(len(x) for row in res for x in row) if res else 1
    for row in res:
        print(" ".join((x if x else ".").rjust(w) for x in row))

def check_layout(root, res):
    h = tree_height(root)
    m, n = h + 1, 2 ** (h + 1) - 1
    assert len(res) == m, ("số hàng", len(res), m)
    assert all(len(row) == n for row in res), ("số cột", [len(r) for r in res], n)
    assert all(isinstance(x, str) for row in res for x in row), "mọi ô phải là chuỗi"
    expected_cells = {}
    def walk(node, r, c):
        if node is None:
            return
        assert (r, c) not in expected_cells, ("hai nút trùng ô", r, c)
        expected_cells[(r, c)] = str(node.val)
        step = 2 ** (h - r - 1) if r < h else 0
        walk(node.left, r + 1, c - step)
        walk(node.right, r + 1, c + step)
    walk(root, 0, (n - 1) // 2)
    for r in range(m):
        for c in range(n):
            want = expected_cells.get((r, c), "")
            assert res[r][c] == want, ("ô sai", r, c, res[r][c], want)
    return True

show(examples[1][1])
print(check_layout(build_tree([1, 2, 3, None, 4]), examples[1][1]))

. . . 1 . . .
. 2 . . . 3 .
. . 4 . . . .
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét cây `[1, 2, 3, None, 4]`:

```
      1
     / \
    2   3
     \
      4
```

**Bước 1 — chiều cao.** Đường dài nhất `1 → 2 → 4` có `2` cạnh → `height = 2`.

**Bước 2 — kích thước bảng.** `m = height + 1 = 3` hàng; `n = 2^(2+1) − 1 = 7` cột.

**Bước 3 — đặt gốc.** Hàng `0`, cột `(7 − 1)/2 = 3`.

**Bước 4 — đặt con.** Độ lệch ở hàng `r` là `2^(height − r − 1)`:

- Từ hàng `0`: độ lệch `2^(2−0−1) = 2`. Con trái `2` ở `(1, 3 − 2) = (1, 1)`; con phải `3` ở `(1, 3 + 2) = (1, 5)`.
- Từ hàng `1`: độ lệch `2^(2−1−1) = 1`. Nút `2` chỉ có con **phải** `4` ở `(2, 1 + 1) = (2, 2)`. Nút `3` không có con.

Hàng cuối `r = height` không có con nên không cần độ lệch.

**Vì sao độ lệch giảm một nửa?** Bảng có `2^(h+1) − 1` cột, tức là "đủ chỗ" cho một cây **đầy** có `h + 1` tầng. Nút gốc chia bảng thành hai nửa bằng nhau, mỗi nửa rộng `2^h − 1` cột và chứa một cây đầy cao `h − 1`. Tâm của nửa trái cách tâm gốc `2^(h−1)` cột; tâm nửa phải cũng vậy. Lặp lại cho các nửa nhỏ hơn: độ lệch thành `2^(h−2)`, `2^(h−3)`, ..., `1`. Vì thế mọi nút có vị trí hoàn toàn xác định và **các nút khác nhau không bao giờ đụng ô nhau**.

In [8]:
h = 2
n = 2 ** (h + 1) - 1
print("bảng", h + 1, "x", n)
c0 = (n - 1) // 2
print("gốc ở cột", c0)
for r in range(h):
    print(f"hàng {r}: độ lệch tới con = 2^({h}-{r}-1) = {2 ** (h - r - 1)}")
print("con của gốc ở cột", c0 - 2 ** (h - 1), "và", c0 + 2 ** (h - 1))

bảng 3 x 7
gốc ở cột 3
hàng 0: độ lệch tới con = 2^(2-0-1) = 2
hàng 1: độ lệch tới con = 2^(2-1-1) = 1
con của gốc ở cột 1 và 5


## 5. Cách 1 — Brute Force / baseline: ghép các bảng con (đệ quy chia đôi)

**Ý tưởng trực tiếp theo cấu trúc "bảng = gốc + bảng trái + bảng phải":**

1. Với một nút trong một **khung** có chiều cao `d` (số hàng `d + 1`), bảng của nó gồm: hàng trên cùng chứa nút ở **giữa**; bên dưới là bảng con trái (khung `d − 1`), một **cột trống** ở giữa, và bảng con phải (khung `d − 1`), ghép **cạnh nhau theo từng hàng**.
2. Con vắng mặt (`None`) cũng được "vẽ" thành bảng **rỗng** cùng kích thước (khung `d − 1`), để mọi bảng con có đúng cỡ.
3. Khung chiều cao `0`: bảng `1 × 1` chứa `str(val)` hoặc `""`.

Cách này dễ đúng vì **dựa vào cấu trúc đệ quy của chính bảng**, không cần công thức cột. Ta dùng nó làm một oracle độc lập với công thức độ lệch.

In [9]:
def print_compose(root):
    h = tree_height(root)

    def render(node, d):
        """Bảng (d+1) x (2^(d+1)-1) cho cây con `node` trong khung cao d (node có thể là None)."""
        if d == 0:
            return [[str(node.val) if node else ""]]
        left = render(node.left if node else None, d - 1)
        right = render(node.right if node else None, d - 1)
        w = len(left[0])
        top = [""] * w + [str(node.val) if node else ""] + [""] * w
        return [top] + [l + [""] + r for l, r in zip(left, right)]

    return render(root, h)

for vals, expected in examples:
    got = print_compose(build_tree(vals))
    assert got == expected, (vals, got)
print("cách 1 khớp 4 ví dụ")
show(print_compose(build_tree([1, 2, 3, None, 4])))

cách 1 khớp 4 ví dụ
. . . 1 . . .
. 2 . . . 3 .
. . 4 . . . .


**Độ phức tạp.** Bảng có `(h+1)(2^(h+1) − 1)` ô. Mỗi mức đệ quy tạo và sao chép lại bảng con; ta "vẽ" cả những nhánh rỗng đủ kích thước.

- **Time:** `O(h · 2^h)` (tương đương kích thước bảng); với `h ≤ 10` là cỡ vài chục nghìn ô.
- **Space:** `O(h · 2^h)` cho bảng cộng các bảng con tạm thời.

**Nút thắt:** ta sao chép các hàng nhiều lần (mỗi tầng ghép lại) và vẽ cả các nhánh rỗng. Có thể **đặt thẳng mỗi nút vào ô cuối cùng của nó** nếu biết trước kích thước bảng và công thức cột.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** kích thước bảng và vị trí của mỗi nút đều xác định bởi **chiều cao `h`** và **vị trí của cha**:

- Biết `h` → biết `m = h + 1`, `n = 2^(h+1) − 1` → tạo bảng toàn `""` một lần.
- Biết ô `(r, c)` của cha → con trái ở `(r + 1, c − 2^(h−r−1))`, con phải ở `(r + 1, c + 2^(h−r−1))`.

Do đó chỉ cần **hai lượt duyệt cây**:

1. Lượt 1: tính `h` (đệ quy hoặc BFS đếm tầng).
2. Lượt 2: duyệt cây (DFS hoặc BFS) mang theo `(r, c)`, đặt `res[r][c] = str(node.val)`.

Mỗi nút được đặt **đúng một lần**; không sao chép bảng.

Ta thử công thức bằng tay-code trước khi viết hàm hoàn chỉnh: đặt từng nút của cây ví dụ vào bảng.

In [10]:
t = build_tree([1, 2, 3, None, 4])
h = tree_height(t)
m, n = h + 1, 2 ** (h + 1) - 1
res = [[""] * n for _ in range(m)]
res[0][(n - 1) // 2] = str(t.val)                         # gốc
res[1][3 - 2] = str(t.left.val)                            # con trái của gốc: cột 3 - 2^(2-0-1)
res[1][3 + 2] = str(t.right.val)                           # con phải của gốc
res[2][1 + 1] = str(t.left.right.val)                      # con phải của nút 2: cột 1 + 2^(2-1-1)
show(res)
assert res == examples[1][1]

. . . 1 . . .
. 2 . . . 3 .
. . 4 . . . .


## 7. Các cách tiếp theo

### Cách 2 — Hai lượt DFS: đo chiều cao rồi đặt bằng công thức độ lệch (cách chuẩn)

**Giải thích từng đoạn:**

1. `h = tree_height(root)` (DFS lượt 1).
2. `m, n = h + 1, 2 ** (h + 1) − 1`; `res = [[""] * n for _ in range(m)]`.
3. `dfs(node, r, c)`: nếu `node` rỗng thì dừng; đặt `res[r][c] = str(node.val)`; gọi `dfs(node.left, r + 1, c − 2^(h−r−1))` và `dfs(node.right, r + 1, c + 2^(h−r−1))`.
4. Gọi `dfs(root, 0, (n − 1) // 2)`.

Lưu ý: lời gọi `dfs` cho con của nút ở hàng cuối `r = h` không xảy ra vì nút hàng cuối là lá (không có con) nên không bao giờ tính mũ âm. Ta tính độ lệch **chỉ khi có con**.

In [11]:
def print_dfs(root):
    h = tree_height(root)
    m, n = h + 1, 2 ** (h + 1) - 1
    res = [[""] * n for _ in range(m)]

    def dfs(node, r, c):
        if node is None:
            return
        res[r][c] = str(node.val)
        if node.left or node.right:                  # có con => r < h, mũ không âm
            step = 2 ** (h - r - 1)
            dfs(node.left, r + 1, c - step)
            dfs(node.right, r + 1, c + step)

    dfs(root, 0, (n - 1) // 2)
    return res

for vals, expected in examples:
    got = print_dfs(build_tree(vals))
    assert got == expected, (vals, got)
print("cách 2 khớp 4 ví dụ")
show(print_dfs(build_tree([1, 2, 3, None, 4])))

cách 2 khớp 4 ví dụ
. . . 1 . . .
. 2 . . . 3 .
. . 4 . . . .


In [12]:
class Solution:
    def printTree(self, root: Optional[TreeNode]) -> List[List[str]]:
        def height(node):
            if node is None:
                return -1
            return 1 + max(height(node.left), height(node.right))

        def dfs(node, r, c):
            if node is None:
                return
            res[r][c] = str(node.val)
            if node.left or node.right:
                step = 2 ** (h - r - 1)
                dfs(node.left, r + 1, c - step)
                dfs(node.right, r + 1, c + step)

        h = height(root)
        m, n = h + 1, 2 ** (h + 1) - 1
        res = [[""] * n for _ in range(m)]
        dfs(root, 0, (n - 1) // 2)
        return res

print([Solution().printTree(build_tree(v)) == e for v, e in examples])

[True, True, True, True]


**Độ phức tạp (cách 2):**

- **Time:** `O(n_nút)` cho hai lượt duyệt + `O(h · 2^h)` để **tạo bảng**. Vì kích thước đầu ra đã là `Θ(h · 2^h)`, đây là tối ưu theo đầu ra.
- **Space:** `O(h · 2^h)` cho bảng kết quả; ngăn xếp đệ quy `O(h)` (`h ≤ 10`, rất nhỏ).

### Cách 3 — BFS lặp (không đệ quy), mang theo `(node, r, c)`

**Giải thích từng đoạn:**

1. **Đo chiều cao bằng BFS:** đếm số tầng khi duyệt theo tầng; `h = số_tầng − 1`.
2. Tạo bảng như cách 2.
3. Hàng đợi chứa bộ ba `(node, r, c)`; lấy ra, ghi `res[r][c]`, rồi đẩy con trái `(node.left, r+1, c−step)` và con phải `(node.right, r+1, c+step)` nếu tồn tại. Độ lệch `step = 1 << (h − r − 1)` chỉ tính khi nút có con (khi đó `r < h`, mũ không âm).

In [13]:
def print_bfs(root):
    # lượt 1: đếm số tầng
    levels, q = 0, deque([root])
    while q:
        levels += 1
        for _ in range(len(q)):
            node = q.popleft()
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
    h = levels - 1

    # lượt 2: đặt từng nút
    m, n = h + 1, 2 ** (h + 1) - 1
    res = [[""] * n for _ in range(m)]
    q = deque([(root, 0, (n - 1) // 2)])
    while q:
        node, r, c = q.popleft()
        res[r][c] = str(node.val)
        if node.left or node.right:
            step = 1 << (h - r - 1)
            if node.left:
                q.append((node.left, r + 1, c - step))
            if node.right:
                q.append((node.right, r + 1, c + step))
    return res

for vals, expected in examples:
    got = print_bfs(build_tree(vals))
    assert got == expected, (vals, got)
print("cách 3 khớp 4 ví dụ")

cách 3 khớp 4 ví dụ


In [14]:
class Solution:
    def printTree(self, root: Optional[TreeNode]) -> List[List[str]]:
        levels, q = 0, deque([root])
        while q:
            levels += 1
            for _ in range(len(q)):
                node = q.popleft()
                if node.left:
                    q.append(node.left)
                if node.right:
                    q.append(node.right)
        h = levels - 1
        m, n = h + 1, 2 ** (h + 1) - 1
        res = [[""] * n for _ in range(m)]
        q = deque([(root, 0, (n - 1) // 2)])
        while q:
            node, r, c = q.popleft()
            res[r][c] = str(node.val)
            if node.left or node.right:
                step = 1 << (h - r - 1)
                if node.left:
                    q.append((node.left, r + 1, c - step))
                if node.right:
                    q.append((node.right, r + 1, c + step))
        return res

print([Solution().printTree(build_tree(v)) == e for v, e in examples])

[True, True, True, True]


**Độ phức tạp (cách 3):**

- **Time:** `O(số_nút)` duyệt + `O(h · 2^h)` tạo bảng.
- **Space:** `O(h · 2^h)` cho bảng; hàng đợi `O(chiều rộng cây)` ≤ `O(2^h)`.

Ưu điểm: **không đệ quy**, nên không lo giới hạn độ sâu (dù với `h ≤ 10` chuyện đó không xảy ra).

### Cách 4 — Công thức đóng: cột theo **chỉ số đống (heap index)**

**Ý tưởng khác hẳn:** thay vì mang theo cột `c` và trừ/cộng độ lệch, ta đánh số vị trí của nút **trong cây đầy** bằng `p` = số thứ tự **trong tầng** (từ `0`, trái sang phải): gốc `p = 0`; con trái của nút `p` có `2p`, con phải có `2p + 1`. Cột được tính **trực tiếp** bởi một công thức:

```
cột(r, p) = (2p + 1) · 2^(h − r) − 1
```

**Kiểm tra:** gốc `r = 0, p = 0` → `2^h − 1 = (n − 1)/2` ✓. Con trái của gốc: `r = 1, p = 0` → `2^(h−1) − 1 = (2^h − 1) − 2^(h−1)` ✓ (lệch `−2^(h−1)`). Đây chính là **thứ tự duyệt giữa (in-order) trong cây đầy**: nút ở tầng `r` vị trí `p` nằm ở vị trí giữa-thứ-tự `(2p+1)·2^(h−r) − 1`.

**Giải thích từng đoạn:**

1. Tính `h` (có thể bằng BFS).
2. Đưa `(node, r, p)` vào hàng đợi; với mỗi nút lấy ra, ghi `res[r][cột(r, p)]`, đẩy con trái với `p' = 2p`, con phải `p' = 2p + 1` ở `r + 1`.
3. Không cần truyền `c`, không cần tính độ lệch riêng từng nút; cột chỉ phụ thuộc `(r, p, h)`.

In [15]:
def print_heap(root):
    levels, q = 0, deque([root])
    while q:
        levels += 1
        for _ in range(len(q)):
            node = q.popleft()
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
    h = levels - 1

    m, n = h + 1, 2 ** (h + 1) - 1
    res = [[""] * n for _ in range(m)]
    q = deque([(root, 0, 0)])
    while q:
        node, r, p = q.popleft()
        res[r][((2 * p + 1) << (h - r)) - 1] = str(node.val)
        if node.left:
            q.append((node.left, r + 1, 2 * p))
        if node.right:
            q.append((node.right, r + 1, 2 * p + 1))
    return res

for vals, expected in examples:
    got = print_heap(build_tree(vals))
    assert got == expected, (vals, got)
print("cách 4 khớp 4 ví dụ")

cách 4 khớp 4 ví dụ


In [16]:
class Solution:
    def printTree(self, root: Optional[TreeNode]) -> List[List[str]]:
        levels, q = 0, deque([root])
        while q:
            levels += 1
            for _ in range(len(q)):
                node = q.popleft()
                if node.left:
                    q.append(node.left)
                if node.right:
                    q.append(node.right)
        h = levels - 1
        m, n = h + 1, 2 ** (h + 1) - 1
        res = [[""] * n for _ in range(m)]
        q = deque([(root, 0, 0)])
        while q:
            node, r, p = q.popleft()
            res[r][((2 * p + 1) << (h - r)) - 1] = str(node.val)
            if node.left:
                q.append((node.left, r + 1, 2 * p))
            if node.right:
                q.append((node.right, r + 1, 2 * p + 1))
        return res

print([Solution().printTree(build_tree(v)) == e for v, e in examples])

[True, True, True, True]


**Độ phức tạp (cách 4):**

- **Time:** `O(số_nút)` duyệt + `O(h · 2^h)` tạo bảng.
- **Space:** `O(h · 2^h)` cho bảng; hàng đợi `O(2^h)`.

Cách này tương đương cách 3 về chi phí nhưng cho thấy **cột là hàm đóng của `(r, p)`**, rất hữu ích cho các bài tương tự (ví dụ *Maximum Width of Binary Tree*, mục 15).

## 8. TRACE từng bước

### 8.1. Trace cách 2 (DFS) trên `[1, 2, 3, None, 4]`

`h = 2`, bảng `3 × 7`, gốc ở cột `3`.

| Bước | Nút | `(r, c)` | Có con? | Độ lệch `2^(h−r−1)` | Con trái → | Con phải → |
|---|---|---|---|---|---|---|
| 1 | `1` | `(0, 3)` | trái, phải | `2^(2−0−1) = 2` | `(1, 1)` | `(1, 5)` |
| 2 | `2` | `(1, 1)` | phải | `2^(2−1−1) = 1` | không có | `(2, 2)` |
| 3 | `4` | `(2, 2)` | lá | — | — | — |
| 4 | `3` | `(1, 5)` | lá | — | — | — |

Thứ tự DFS: `1, 2, 4, 3`. Bảng cuối cùng khớp ví dụ 2.

In [17]:
def trace_dfs(root):
    h = tree_height(root)
    m, n = h + 1, 2 ** (h + 1) - 1
    res = [[""] * n for _ in range(m)]
    print(f"h = {h}, bảng {m} x {n}, gốc ở cột {(n - 1) // 2}")
    step_no = 0

    def dfs(node, r, c, tag):
        nonlocal step_no
        if node is None:
            return
        step_no += 1
        res[r][c] = str(node.val)
        line = f"bước {step_no}: {tag:5} nút {node.val:>3} đặt tại ({r}, {c})"
        if node.left or node.right:
            step = 2 ** (h - r - 1)
            line += f" | độ lệch 2^({h}-{r}-1) = {step}"
            print(line)
            dfs(node.left, r + 1, c - step, "trái")
            dfs(node.right, r + 1, c + step, "phải")
        else:
            print(line + " | lá, không có con")

    dfs(root, 0, (n - 1) // 2, "gốc")
    return res

res = trace_dfs(build_tree([1, 2, 3, None, 4]))
show(res)

h = 2, bảng 3 x 7, gốc ở cột 3
bước 1: gốc   nút   1 đặt tại (0, 3) | độ lệch 2^(2-0-1) = 2
bước 2: trái  nút   2 đặt tại (1, 1) | độ lệch 2^(2-1-1) = 1
bước 3: phải  nút   4 đặt tại (2, 2) | lá, không có con
bước 4: phải  nút   3 đặt tại (1, 5) | lá, không có con
. . . 1 . . .
. 2 . . . 3 .
. . 4 . . . .


### 8.2. Trace cách 4 (chỉ số đống) trên cây `[1, 2, 3, 4, 5, 6, 7]` (cây đầy)

Với cây đầy `h = 2`, cột của `(r, p)` là `(2p + 1)·2^(2−r) − 1`:

| Tầng `r` | `p` | công thức | cột |
|---|---|---|---|
| 0 | 0 | `(1)·4 − 1` | 3 |
| 1 | 0 | `(1)·2 − 1` | 1 |
| 1 | 1 | `(3)·2 − 1` | 5 |
| 2 | 0 | `(1)·1 − 1` | 0 |
| 2 | 1 | `(3)·1 − 1` | 2 |
| 2 | 2 | `(5)·1 − 1` | 4 |
| 2 | 3 | `(7)·1 − 1` | 6 |

Hàng cuối chiếm đủ các cột `0, 2, 4, 6`: mỗi lá cách nhau `2` cột — cách duy nhất để một cây đầy vừa vặn trong `7` cột.

In [18]:
def trace_heap(root):
    levels, q = 0, deque([root])
    while q:
        levels += 1
        for _ in range(len(q)):
            node = q.popleft()
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
    h = levels - 1
    m, n = h + 1, 2 ** (h + 1) - 1
    res = [[""] * n for _ in range(m)]
    q = deque([(root, 0, 0)])
    while q:
        node, r, p = q.popleft()
        col = ((2 * p + 1) << (h - r)) - 1
        res[r][col] = str(node.val)
        print(f"nút {node.val:>3}: tầng r={r}, p={p} -> cột ({2 * p + 1}<<{h - r}) - 1 = {col}")
        if node.left:
            q.append((node.left, r + 1, 2 * p))
        if node.right:
            q.append((node.right, r + 1, 2 * p + 1))
    return res

res = trace_heap(build_tree([1, 2, 3, 4, 5, 6, 7]))
show(res)

nút   1: tầng r=0, p=0 -> cột (1<<2) - 1 = 3
nút   2: tầng r=1, p=0 -> cột (1<<1) - 1 = 1
nút   3: tầng r=1, p=1 -> cột (3<<1) - 1 = 5
nút   4: tầng r=2, p=0 -> cột (1<<0) - 1 = 0
nút   5: tầng r=2, p=1 -> cột (3<<0) - 1 = 2
nút   6: tầng r=2, p=2 -> cột (5<<0) - 1 = 4
nút   7: tầng r=2, p=3 -> cột (7<<0) - 1 = 6
. . . 1 . . .
. 2 . . . 3 .
4 . 5 . 6 . 7


## 9. Code nộp LeetCode

Đây là cách 2 (hai lượt DFS) — đúng theo đề, ngắn và dễ đọc. Khi nộp lên LeetCode không cần định nghĩa `TreeNode`, và `Optional`, `List` đã được import sẵn; ở đây ta giữ lại ở đầu notebook để chạy cục bộ.

In [19]:
class Solution:
    def printTree(self, root: Optional[TreeNode]) -> List[List[str]]:
        def height(node):
            if node is None:
                return -1
            return 1 + max(height(node.left), height(node.right))

        def dfs(node, r, c):
            if node is None:
                return
            res[r][c] = str(node.val)
            if node.left or node.right:
                step = 2 ** (h - r - 1)
                dfs(node.left, r + 1, c - step)
                dfs(node.right, r + 1, c + step)

        h = height(root)
        m, n = h + 1, 2 ** (h + 1) - 1
        res = [[""] * n for _ in range(m)]
        dfs(root, 0, (n - 1) // 2)
        return res


sol = Solution()
for vals, expected in examples:
    root = build_tree(vals)
    got = sol.printTree(root)
    assert got == expected, vals
    assert check_layout(root, got)
    print(vals, "-> OK")
print("Chạy ví dụ: OK")

[1, 2] -> OK
[1, 2, 3, None, 4] -> OK
[5] -> OK
[1, None, 2] -> OK
Chạy ví dụ: OK


## 10. Vì sao đúng

**Tính hợp lệ của kích thước và vị trí.**

1. **Đủ chỗ:** nút ở hàng `r` có cột `c` trong khoảng `[0, n−1]`. Quy nạp theo `r`: gốc ở cột `2^h − 1`, độ lệch `2^(h−1)` đưa các con vào `[2^(h−1) − 1, 3·2^(h−1) − 1]` — nằm trong bảng. Tổng quát nút ở hàng `r` có cột dạng `(2p + 1)·2^(h−r) − 1`, với `0 ≤ p ≤ 2^r − 1` (cách 4), tức trong `[2^(h−r) − 1, n − 2^(h−r)]` ⊂ `[0, n−1]`.
2. **Không trùng ô:** hai nút khác nhau ở **cùng hàng** `r` có `p` khác nhau nên cột `(2p+1)·2^(h−r) − 1` khác nhau (hàm tăng chặt theo `p`). Hai nút **khác hàng** đương nhiên khác ô.
3. **Công thức độ lệch khớp công thức đóng:** nếu cha có `(r, p)` và cột `c = (2p+1)·2^(h−r) − 1` thì con trái có `p' = 2p`, cột `(4p+1)·2^(h−r−1) − 1 = c − 2^(h−r−1)` vì `(2p+1)·2^(h−r) = (4p+2)·2^(h−r−1)`, hiệu bằng `2^(h−r−1)`. Tương tự con phải `p' = 2p+1` cho `c + 2^(h−r−1)`.
4. **Mũ không âm:** độ lệch chỉ được tính cho nút có con; nút có con ở hàng `r` thì `r < h` (vì `h` là chiều cao lớn nhất), nên `h − r − 1 ≥ 0`.

Vậy cách 2, 3 và 4 đều đặt đúng từng nút vào đúng ô quy định bởi đề, và mọi ô còn lại giữ `""` do ta khởi tạo.

Ta kiểm chứng bằng code: cả bốn cách cho **cùng một bảng** và bảng thoả `check_layout` (độc lập, đi theo đúng quy tắc của đề) trên nhiều cây ngẫu nhiên.

In [20]:
def random_small_tree(rng, max_nodes=40, max_height=9):
    while True:
        n = rng.randint(1, max_nodes)
        vals = random_tree_list(rng, n, -99, 99)
        root = build_tree(vals)
        if tree_height(root) <= max_height:
            return vals, root

rng = random.Random(655)
count = 0
for _ in range(400):
    vals, root = random_small_tree(rng)
    tables = [print_compose(root), print_dfs(root), print_bfs(root), print_heap(root)]
    for tb in tables:
        assert check_layout(root, tb), vals
    assert all(tb == tables[0] for tb in tables), vals
    count += 1
print("đã kiểm chứng 4 cách cho cùng bảng và thoả check_layout trên", count, "cây ngẫu nhiên")

đã kiểm chứng 4 cách cho cùng bảng và thoả check_layout trên 400 cây ngẫu nhiên


## 11. Time / Space Complexity

Gọi `N` là số nút, `h` là chiều cao. Kích thước đầu ra là `S = (h + 1)(2^(h+1) − 1) = Θ(h · 2^h)` ô.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Ghép bảng con | `O(S)` (nhưng sao chép nhiều lần, hằng số lớn) | `O(S)` + bảng tạm | oracle độc lập với công thức |
| 2. Hai lượt DFS + độ lệch | `O(N + S)` | `O(S)` + ngăn xếp `O(h)` | **cách nộp chuẩn** |
| 3. BFS lặp | `O(N + S)` | `O(S)` + hàng đợi `O(2^h)` | không đệ quy |
| 4. Chỉ số đống (công thức đóng) | `O(N + S)` | `O(S)` + hàng đợi `O(2^h)` | cột là hàm đóng của `(r, p)` |

Vì `N ≤ 2^10` và `S ≤ 11 · 2047 ≈ 22 500` nên mọi cách đều rất nhanh. Chi phí chủ yếu là **tạo bảng**: không thể nhanh hơn kích thước đầu ra `Θ(S)`.

Chú ý khác biệt `N` và `S`: một cây lệch `N = 11` nút vẫn có `S ≈ 22 500` ô, nên không thể coi thời gian chỉ là `O(N)`.

## 12. Bộ test

Gồm: ví dụ đề (so sánh **bảng** đúng như đề), edge case (một nút, cây lệch trái/phải, cây đầy, giá trị âm/`0`/hai chữ số), phản ví dụ cho các cách làm sai, **hàm kiểm tra bố cục** `check_layout` theo đúng quy tắc của đề, đối chiếu bốn cách với nhau trên nhiều cây ngẫu nhiên có seed (đúng ràng buộc giá trị và chiều cao), và các test lớn nhất theo ràng buộc.

In [21]:
METHODS = {
    "compose": print_compose,
    "dfs": print_dfs,
    "bfs": print_bfs,
    "heap": print_heap,
}

def check_all(vals, expected=None):
    root = build_tree(vals)
    ref = None
    for name, fn in METHODS.items():
        got = fn(root)
        assert check_layout(root, got), (name, vals)
        if expected is not None:
            assert got == expected, (name, vals, got)
        if ref is None:
            ref = got
        assert got == ref, (name, vals)
    return ref

for vals, expected in examples:
    check_all(vals, expected)

# edge case
check_all([0], [["0"]])                                     # giá trị 0 là chuỗi "0", không phải ""
check_all([-99], [["-99"]])
check_all([1, 2, 3], [["", "1", ""], ["2", "", "3"]])
check_all([1, 2, None, 3], [["", "", "", "1", "", "", ""],
                            ["", "2", "", "", "", "", ""],
                            ["3", "", "", "", "", "", ""]])        # lệch trái 3 nút
check_all([1, None, 2, None, 3], [["", "", "", "1", "", "", ""],
                                  ["", "", "", "", "", "2", ""],
                                  ["", "", "", "", "", "", "3"]])  # lệch phải 3 nút
full7 = check_all([1, 2, 3, 4, 5, 6, 7], [["", "", "", "1", "", "", ""],
                                          ["", "2", "", "", "", "3", ""],
                                          ["4", "", "5", "", "6", "", "7"]])
check_all([99, -99, 0, None, 5, 10])                        # trộn âm, 0, hai chữ số
print("edge case: OK")
show(full7)

edge case: OK
. . . 1 . . .
. 2 . . . 3 .
4 . 5 . 6 . 7


**Phản ví dụ** cho các cách làm sai điển hình (chi tiết ở mục 13):

- Dùng `[[""] * n] * m`: các hàng **dùng chung** một list nên cây `[1, 2]` cho kết quả sai.
- Độ lệch `2 ** (h − r)` (thiếu `− 1`): chỉ số **âm** ghi âm thầm vào cuối hàng, không báo lỗi.
- Chiều cao tính bằng số **nút** trên đường dài nhất: bảng to gấp đôi và gốc sai chỗ.

In [22]:
# Đối chiếu bốn cách trên nhiều cây ngẫu nhiên (seed cố định); giá trị trong [-99, 99]
rng = random.Random(2026)
count = 0
for _ in range(800):
    vals, root = random_small_tree(rng, max_nodes=rng.choice([3, 8, 20, 60]), max_height=9)
    ref = print_compose(root)
    for name, fn in METHODS.items():
        got = fn(root)
        assert got == ref, (name, vals)
    assert check_layout(root, ref), vals
    count += 1
print("đã đối chiếu", count, "cây ngẫu nhiên: OK")

# kiểm tra kích thước bảng theo công thức của đề
def tree_to_list_nodes(root):
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        if node:
            out.append(node.val)
            stack += [node.left, node.right]
    return out

for _ in range(100):
    vals, root = random_small_tree(rng, 50, 9)
    h = tree_height(root)
    res = print_dfs(root)
    assert len(res) == h + 1 and all(len(r) == 2 ** (h + 1) - 1 for r in res)
    nonempty = sum(1 for row in res for x in row if x != "")
    assert nonempty == len(tree_to_list_nodes(root))          # số ô khác rỗng = số nút
print("kích thước bảng: OK")

đã đối chiếu 800 cây ngẫu nhiên: OK
kích thước bảng: OK


In [23]:
# Test lớn nhất theo ràng buộc
def skew(h, left=True, rng=random.Random(5)):
    root = TreeNode(rng.randint(-99, 99))
    cur = root
    for _ in range(h):
        node = TreeNode(rng.randint(-99, 99))
        if left:
            cur.left = node
        else:
            cur.right = node
        cur = node
    return root

def perfect(h, rng=random.Random(6)):
    vals = [rng.randint(-99, 99) for _ in range(2 ** (h + 1) - 1)]
    return build_tree(vals)

def zigzag(h, rng=random.Random(7)):
    root = TreeNode(rng.randint(-99, 99))
    cur, left = root, True
    for _ in range(h):
        node = TreeNode(rng.randint(-99, 99))
        if left:
            cur.left = node
        else:
            cur.right = node
        cur, left = node, not left
    return root

big = {
    "lệch trái h=10": skew(10, True),
    "lệch phải h=10": skew(10, False),
    "zigzag h=10": zigzag(10),
    "đầy h=9 (1023 nút)": perfect(9),
}
for name, root in big.items():
    row, ref = [], None
    for mname, fn in METHODS.items():
        t0 = time.perf_counter()
        res = fn(root)
        dt = (time.perf_counter() - t0) * 1000
        assert check_layout(root, res), (name, mname)
        if ref is None:
            ref = res
        assert res == ref, (name, mname)
        row.append(f"{mname} {dt:6.2f} ms")
    print(f"{name:20} bảng {len(ref)} x {len(ref[0])} |", " | ".join(row))

lệch trái h=10       bảng 11 x 2047 | compose   1.53 ms | dfs   0.02 ms | bfs   0.19 ms | heap   0.12 ms
lệch phải h=10       bảng 11 x 2047 | compose   1.24 ms | dfs   0.02 ms | bfs   0.03 ms | heap   0.12 ms
zigzag h=10          bảng 11 x 2047 | compose   1.26 ms | dfs   0.03 ms | bfs   0.02 ms | heap   0.11 ms
đầy h=9 (1023 nút)   bảng 10 x 1023 | compose   0.87 ms | dfs   0.36 ms | bfs   0.85 ms | heap   0.38 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tạo ma trận bằng `[[""] * n] * m`

**Code sai:**

```python
res = [[""] * n] * m          # m tham chiếu tới CÙNG một list hàng
```

**Vì sao sai:** `[""] * n` tạo **một** list; `* m` chỉ **lặp lại tham chiếu** tới nó. Ghi vào `res[1][0]` thì mọi hàng đều đổi. Phải dùng `[[""] * n for _ in range(m)]`.

In [24]:
def wrong_alias(root):
    h = tree_height(root)
    m, n = h + 1, 2 ** (h + 1) - 1
    res = [[""] * n] * m                              # SAI
    def dfs(node, r, c):
        if node is None:
            return
        res[r][c] = str(node.val)
        if node.left or node.right:
            step = 2 ** (h - r - 1)
            dfs(node.left, r + 1, c - step)
            dfs(node.right, r + 1, c + step)
    dfs(root, 0, (n - 1) // 2)
    return res

t = build_tree([1, 2])
print("sai :", wrong_alias(t))
print("đúng:", print_dfs(t))

sai : [['2', '1', ''], ['2', '1', '']]
đúng: [['', '1', ''], ['2', '', '']]


### Lỗi 2: Tính chiều cao bằng số **nút** thay vì số **cạnh**

**Code sai:**

```python
def height(node):
    if node is None:
        return 0                # cây rỗng cao 0 -> lá cao 1, đếm NÚT
    return 1 + max(height(node.left), height(node.right))
```

**Vì sao sai:** đề định nghĩa `height` bằng số **cạnh**: cây một nút cao `0`. Hàm trên cho cây một nút cao `1` nên bảng có `h + 1` hàng nhưng **dư một hàng** và `n` **gấp đôi**, gốc nằm lệch chỗ.

In [25]:
def wrong_height_count_nodes(root):
    def height(node):
        if node is None:
            return 0                                  # SAI: nên là -1
        return 1 + max(height(node.left), height(node.right))
    h = height(root)
    m, n = h + 1, 2 ** (h + 1) - 1
    res = [[""] * n for _ in range(m)]
    res[0][(n - 1) // 2] = str(root.val)
    return res

print("sai :", wrong_height_count_nodes(build_tree([5])), "(phải là [['5']])")
print("đúng:", print_dfs(build_tree([5])))

sai : [['', '5', ''], ['', '', '']] (phải là [['5']])
đúng: [['5']]


### Lỗi 3: Độ lệch `2 ** (h − r)` thay vì `2 ** (h − r − 1)`

**Code sai:**

```python
step = 2 ** (h - r)           # thiếu - 1
```

**Vì sao sai:** gốc ở hàng `r = 0` phải lệch `2^(h−1)`, không phải `2^h`. Lệch quá tay làm con trái rơi ra ngoài bảng: chỉ số **âm** như `res[1][-1]` **không báo lỗi** mà âm thầm ghi vào **cuối hàng**, kết quả sai mà không có `IndexError`.

In [26]:
def wrong_step(root):
    h = tree_height(root)
    m, n = h + 1, 2 ** (h + 1) - 1
    res = [[""] * n for _ in range(m)]
    def dfs(node, r, c):
        if node is None:
            return
        res[r][c] = str(node.val)
        if node.left or node.right:
            step = 2 ** (h - r)                       # SAI
            dfs(node.left, r + 1, c - step)
            dfs(node.right, r + 1, c + step)
    dfs(root, 0, (n - 1) // 2)
    return res

t = build_tree([1, 2])
try:
    print("sai :", wrong_step(t))        # con trái ghi vào cột -1 (cuối hàng), con phải vượt biên
except IndexError as e:
    print("sai : IndexError:", e)
print("đúng:", print_dfs(t))

sai : [['', '1', ''], ['', '', '2']]
đúng: [['', '1', ''], ['2', '', '']]


### Lỗi 4: Quên `str()`

**Code sai:**

```python
res[r][c] = node.val          # số nguyên, không phải chuỗi
```

**Vì sao sai:** đề yêu cầu ma trận **chuỗi** (`List[List[str]]`). Ô chứa số nguyên không bằng chuỗi: `1 != "1"`, nên bị chấm sai dù bố cục đúng.

In [27]:
def wrong_no_str(root):
    res = print_dfs(root)
    h = tree_height(root)
    n = 2 ** (h + 1) - 1
    res[0][(n - 1) // 2] = root.val                    # SAI: số nguyên
    return res

t = build_tree([1, 2])
print("sai :", wrong_no_str(t), "| bằng đáp án:", wrong_no_str(t) == examples[0][1])
print("đúng:", print_dfs(t), "| bằng đáp án:", print_dfs(t) == examples[0][1])

sai : [['', 1, ''], ['2', '', '']] | bằng đáp án: False
đúng: [['', '1', ''], ['2', '', '']] | bằng đáp án: True


### Lỗi 5: Bỏ sót nút có giá trị `0` vì kiểm tra "thật/giả" của giá trị

**Code sai:**

```python
if node.val:                  # giá trị 0 bị coi là "giả" => không đặt
    res[r][c] = str(node.val)
```

**Vì sao sai:** `0` là giá trị hợp lệ (`-99 <= val <= 99`) nhưng trong điều kiện `if` nó coi như `False`. Ô tương ứng sẽ để trống `""` — ô trống và nút `0` bị lẫn lộn. Phải kiểm tra **nút** có tồn tại (`node is not None`), không kiểm tra giá trị.

In [28]:
def wrong_zero(root):
    h = tree_height(root)
    m, n = h + 1, 2 ** (h + 1) - 1
    res = [[""] * n for _ in range(m)]
    def dfs(node, r, c):
        if node is None:
            return
        if node.val:                                   # SAI với giá trị 0
            res[r][c] = str(node.val)
        if node.left or node.right:
            step = 2 ** (h - r - 1)
            dfs(node.left, r + 1, c - step)
            dfs(node.right, r + 1, c + step)
    dfs(root, 0, (n - 1) // 2)
    return res

t = build_tree([1, 0])
print("sai :", wrong_zero(t))
print("đúng:", print_dfs(t))

sai : [['', '1', ''], ['', '', '']]
đúng: [['', '1', ''], ['0', '', '']]


## 14. Mẹo & nhận dạng pattern

- **"In cây vào bảng/lưới theo vị trí":** luôn đặt hai câu hỏi — *kích thước bảng* (phụ thuộc chiều cao, không phụ thuộc số nút) và *công thức vị trí con theo vị trí cha*. Khi vị trí hoàn toàn xác định, hai lượt duyệt là đủ.
- **Chiều cao tính bằng cạnh hay nút?** Đọc kỹ định nghĩa trong đề. Quy ước thuận tiện: `height(None) = −1`, `height(lá) = 0`.
- **Độ lệch giảm một nửa theo tầng** (`2^(h−r−1)`): dấu hiệu của bố cục dựa trên **chia đôi**, giống tìm kiếm nhị phân; vị trí cũng chính là thứ tự duyệt giữa trong cây đầy.
- **Chỉ số đống:** `(r, p)` hoặc đánh số `1, 2, 3, ...` (con của `k` là `2k`, `2k+1`) dùng trong *Maximum Width of Binary Tree*, lưu cây bằng mảng, heap.
- **Cẩn thận chỉ số âm trong Python:** `res[r][−1]` không báo lỗi. Khi tính vị trí bằng phép trừ, hãy `assert 0 <= c < n` trong lúc gỡ lỗi.
- **Ma trận 2 chiều:** luôn `[[x] * n for _ in range(m)]`, không bao giờ `[[x] * n] * m`.
- **Test bài "định dạng":** viết hàm kiểm tra theo **luật của đề** (đi dọc cây tính vị trí, so từng ô) thay vì chỉ so với vài ví dụ viết tay.

## 15. Biến thể

1. **In bố cục ra màn hình:** nối từng hàng thành một dòng chữ (ô trống thành khoảng trắng) — dùng hàm `show` / `to_text` bên dưới.
2. **Maximum Width of Binary Tree (662):** độ rộng một tầng là khoảng cách (đo bằng số vị trí trong cây đầy) giữa nút trái nhất và phải nhất cùng tầng; dùng chỉ số đống `p` của cách 4: rộng tầng = `p_cuối − p_đầu + 1`.
3. **In theo chiều dọc/ngang (xoay 90°):** đổi vai trò hàng và cột.
4. **Bố cục "gọn" (compact):** thay vì chiều rộng cố định `2^(h+1) − 1`, tính chiều rộng theo cây con thực tế (nút vắng không chiếm chỗ) — bài khó hơn, cần đo chiều rộng cây con.
5. **Cây N-phân:** đặt các con cách đều quanh cha; công thức độ lệch thay đổi.
6. **Serialize/Deserialize Binary Tree (297):** chuyển cây ↔ chuỗi/list; hàm `tree_to_list` ở đầu notebook là một dạng.

Dưới đây là biến thể 1 và 2:

In [29]:
def to_text(res):
    return "\n".join("".join(x.center(3) if x else "   " for x in row).rstrip() for row in res)

def max_width(root):
    """Biến thể 662: độ rộng lớn nhất của một tầng (đếm cả vị trí trống ở giữa)."""
    best = 0
    q = deque([(root, 0)])
    while q:
        lo, hi = q[0][1], q[-1][1]
        best = max(best, hi - lo + 1)
        for _ in range(len(q)):
            node, p = q.popleft()
            if node.left:
                q.append((node.left, 2 * p))
            if node.right:
                q.append((node.right, 2 * p + 1))
    return best

print(to_text(print_dfs(build_tree([1, 2, 3, 4, None, None, 7, 8]))))
print("max_width:", max_width(build_tree([1, 3, 2, 5, 3, None, 9])))    # 4 (ví dụ của bài 662)
assert max_width(build_tree([1, 3, 2, 5, 3, None, 9])) == 4
assert max_width(build_tree([1, 3, 2, 5])) == 2
assert max_width(build_tree([1, 3, 2, 5, None, None, 9, 6, None, 7])) == 7

                      1
          2                       3
    4                                   7
 8
max_width: 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ghép bảng con | Bảng = gốc + bảng trái + cột trống + bảng phải (đệ quy trên khung cố định) | `O(S)` (hằng số lớn) | `O(S)` + tạm | Oracle độc lập; hiểu cấu trúc đệ quy của bố cục |
| 2. Hai lượt DFS | Đo `h`, rồi đặt nút bằng độ lệch `2^(h−r−1)` | `O(N + S)` | `O(S)` + `O(h)` | **Cách nộp chuẩn**, bám sát đề |
| 3. BFS lặp | Giống cách 2 nhưng dùng hàng đợi `(node, r, c)` | `O(N + S)` | `O(S)` + `O(2^h)` | Khi muốn tránh đệ quy |
| 4. Chỉ số đống | Cột = `(2p + 1)·2^(h−r) − 1` theo `(r, p)` | `O(N + S)` | `O(S)` + `O(2^h)` | Khi cần công thức đóng; liên quan bài *Maximum Width* |

(`N` = số nút, `S = (h+1)(2^(h+1) − 1)` = số ô của bảng.)

Kết luận: nộp **cách 2**. Cách 1 làm oracle độc lập, cách 4 mở rộng sang các bài dùng chỉ số đống.

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có một cell để bạn làm. Các cell TODO bên dưới **chỉ khai báo** hàm rỗng hoặc gán `None`; hãy tự điền rồi bỏ comment dòng kiểm tra. Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Đoán kết quả trước khi chạy (viết ra giấy), rồi chạy để so sánh.

In [30]:
g = [[""] * 2] * 2
g[0][0] = "x"
print(g)
k = [[""] * 2 for _ in range(2)]
k[0][0] = "x"
print(k)
print(tree_height(build_tree([1, 2, 3, None, 4])), tree_height(build_tree([7])))
print(2 ** 3, 1 << 3, (15 - 1) // 2)
print(print_dfs(build_tree([1, None, 2])))

[['x', ''], ['x', '']]
[['x', ''], ['', '']]
2 0
8 8 7
[['', '1', ''], ['', '', '2']]


### Bài 2 (tính tay)

Với cây `[4, 2, 6, 1, None, 5]`, hãy tính `h`, kích thước bảng và toạ độ `(hàng, cột)` của **từng nút**. Gán kết quả vào `my_h` (số), `my_size` (tuple `(m, n)`) và `my_positions` (dict từ giá trị nút sang tuple `(r, c)`; các giá trị trong cây này đều khác nhau). Hiện để `None`.

In [31]:
my_h = None
my_size = None
my_positions = None      # ví dụ: {4: (0, ?), 2: (1, ?), ...}
# Sau khi điền, bỏ comment để kiểm tra:
# t = build_tree([4, 2, 6, 1, None, 5])
# res = print_dfs(t)
# truth = {int(res[r][c]): (r, c) for r in range(len(res)) for c in range(len(res[0])) if res[r][c] != ""}
# assert my_h == tree_height(t) and my_size == (len(res), len(res[0])) and my_positions == truth

### Bài 3 (điền TODO)

Viết `my_height(root)`: trả chiều cao của cây theo số **cạnh** (cây rỗng là `-1`).

In [32]:
def my_height(root):
    # TODO: trường hợp cơ sở cho None, rồi 1 + max(chiều cao con trái, chiều cao con phải)
    pass

# Kiểm tra (bỏ comment sau khi viết xong):
# assert my_height(None) == -1
# assert my_height(build_tree([5])) == 0
# assert my_height(build_tree([1, 2, 3, None, 4])) == 2
# assert my_height(build_tree([1, None, 2, None, 3, None, 4])) == 3

### Bài 4 (tự viết lại)

Viết `print_tree_iter(root)` **không dùng đệ quy** (dùng `deque` hoặc danh sách làm ngăn xếp), trả đúng bảng như đề. Không xem lại code ở trên.

In [33]:
def print_tree_iter(root):
    # TODO
    return None

# for vals, expected in examples:
#     assert print_tree_iter(build_tree(vals)) == expected

### Bài 5 (biến thể / mini challenge)

Viết `layer_widths(root)` trả về **list độ rộng từng tầng** theo nghĩa bài *Maximum Width* (số vị trí từ nút trái nhất tới nút phải nhất cùng tầng, **đếm cả vị trí trống ở giữa**). Hãy kiểm tra bằng một **oracle dựa trên bảng của bài này**: với hàng `r` của bảng, các nút nằm ở những cột không rỗng; hai nút kề nhau trong cây đầy cách nhau `2^(h−r+1)` cột, nên độ rộng tầng = `(cột_cuối − cột_đầu) // 2^(h−r+1) + 1`.

In [34]:
def layer_widths(root):
    # TODO
    return None

def widths_oracle(root):
    res = print_dfs(root)
    h = len(res) - 1
    out = []
    for r, row in enumerate(res):
        cols = [c for c, x in enumerate(row) if x != ""]
        out.append((cols[-1] - cols[0]) // (1 << (h - r + 1)) + 1)
    return out

# for vals in ([1], [1, 3, 2, 5, 3, None, 9], [1, 3, 2, 5], [1, 3, 2, 5, None, None, 9, 6, None, 7]):
#     assert layer_widths(build_tree(vals)) == widths_oracle(build_tree(vals)), vals

## 18. Gợi ý (3 mức)

**Bài 1.**
- Mức 1: `[[""] * 2] * 2` lặp **tham chiếu**; `for _ in range(2)` tạo list mới mỗi lần.
- Mức 2: `[1, 2, 3, None, 4]` có chiều cao `2` (đường `1-2-4`). `(15 − 1) // 2 = 7`.
- Mức 3: kết quả theo thứ tự: `[['x', ''], ['x', '']]`, `[['x', ''], ['', '']]`, `2 0`, `8 8 7`, `[['', '1', ''], ['', '', '2']]`.

**Bài 2.**
- Mức 1: đường dài nhất `4 → 2 → 1` có `2` cạnh nên `h = 2`; bảng `3 × 7`; gốc ở cột `3`.
- Mức 2: độ lệch hàng 0 là `2`, hàng 1 là `1`. `2` ở `(1, 1)`, `6` ở `(1, 5)`.
- Mức 3: `1` là con trái của `2` ở `(2, 0)`; `5` là con trái của `6` ở `(2, 4)`. Vậy `{4: (0, 3), 2: (1, 1), 6: (1, 5), 1: (2, 0), 5: (2, 4)}`.

**Bài 3.**
- Mức 1: `None` trả về `-1`.
- Mức 2: lá có hai con `None` → `1 + max(-1, -1) = 0`.
- Mức 3: `return 1 + max(my_height(root.left), my_height(root.right))`.

**Bài 4.**
- Mức 1: cần hai lượt: một để đo `h`, một để đặt nút. Có thể dùng `deque` cho cả hai.
- Mức 2: lượt đo: đếm số tầng; lượt đặt: hàng đợi `(node, r, c)`.
- Mức 3: chỉ tính `step = 1 << (h - r - 1)` khi nút có con; đặt `res[r][c] = str(node.val)`.

**Bài 5.**
- Mức 1: đánh số nút theo chỉ số trong tầng: gốc `0`, con của `p` là `2p` và `2p + 1`.
- Mức 2: duyệt theo tầng; với mỗi tầng giữ `p` của nút đầu và nút cuối.
- Mức 3: độ rộng tầng = `p_cuối − p_đầu + 1`; thêm vào list sau mỗi tầng.

## 19. Đáp án tham khảo

Các đáp án dưới đây đều chạy kiểm tra bằng `assert`.

In [35]:
# Bài 1
g = [[""] * 2] * 2
g[0][0] = "x"
assert g == [["x", ""], ["x", ""]]
k = [[""] * 2 for _ in range(2)]
k[0][0] = "x"
assert k == [["x", ""], ["", ""]]
assert (tree_height(build_tree([1, 2, 3, None, 4])), tree_height(build_tree([7]))) == (2, 0)
assert (2 ** 3, 1 << 3, (15 - 1) // 2) == (8, 8, 7)
assert print_dfs(build_tree([1, None, 2])) == [["", "1", ""], ["", "", "2"]]
print("Bài 1: [['x', ''], ['x', '']] | [['x', ''], ['', '']] | 2 0 | 8 8 7 | [['', '1', ''], ['', '', '2']]")

# Bài 2
t = build_tree([4, 2, 6, 1, None, 5])
my_h = 2
my_size = (3, 7)
my_positions = {4: (0, 3), 2: (1, 1), 6: (1, 5), 1: (2, 0), 5: (2, 4)}
res = print_dfs(t)
truth = {int(res[r][c]): (r, c) for r in range(len(res)) for c in range(len(res[0])) if res[r][c] != ""}
assert my_h == tree_height(t) and my_size == (len(res), len(res[0])) and my_positions == truth
print("Bài 2: OK")
show(res)

Bài 1: [['x', ''], ['x', '']] | [['x', ''], ['', '']] | 2 0 | 8 8 7 | [['', '1', ''], ['', '', '2']]
Bài 2: OK
. . . 4 . . .
. 2 . . . 6 .
1 . . . 5 . .


In [36]:
# Bài 3
def my_height(root):
    if root is None:
        return -1
    return 1 + max(my_height(root.left), my_height(root.right))

assert my_height(None) == -1
assert my_height(build_tree([5])) == 0
assert my_height(build_tree([1, 2, 3, None, 4])) == 2
assert my_height(build_tree([1, None, 2, None, 3, None, 4])) == 3
print("Bài 3: OK")

# Bài 4
def print_tree_iter(root):
    levels, q = 0, deque([root])
    while q:
        levels += 1
        for _ in range(len(q)):
            node = q.popleft()
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
    h = levels - 1
    m, n = h + 1, 2 ** (h + 1) - 1
    res = [[""] * n for _ in range(m)]
    q = deque([(root, 0, (n - 1) // 2)])
    while q:
        node, r, c = q.popleft()
        res[r][c] = str(node.val)
        if node.left or node.right:
            step = 1 << (h - r - 1)
            if node.left:
                q.append((node.left, r + 1, c - step))
            if node.right:
                q.append((node.right, r + 1, c + step))
    return res

for vals, expected in examples:
    assert print_tree_iter(build_tree(vals)) == expected
rng = random.Random(4)
for _ in range(200):
    vals, root = random_small_tree(rng)
    assert print_tree_iter(root) == print_dfs(root)
print("Bài 4: OK")

Bài 3: OK
Bài 4: OK


In [37]:
# Bài 5
def layer_widths(root):
    out = []
    q = deque([(root, 0)])
    while q:
        out.append(q[-1][1] - q[0][1] + 1)
        for _ in range(len(q)):
            node, p = q.popleft()
            if node.left:
                q.append((node.left, 2 * p))
            if node.right:
                q.append((node.right, 2 * p + 1))
    return out

def widths_oracle(root):
    res = print_dfs(root)
    h = len(res) - 1
    out = []
    for r, row in enumerate(res):
        cols = [c for c, x in enumerate(row) if x != ""]
        out.append((cols[-1] - cols[0]) // (1 << (h - r + 1)) + 1)
    return out

for vals in ([1], [1, 3, 2, 5, 3, None, 9], [1, 3, 2, 5], [1, 3, 2, 5, None, None, 9, 6, None, 7]):
    assert layer_widths(build_tree(vals)) == widths_oracle(build_tree(vals)), vals
rng = random.Random(55)
for _ in range(300):
    vals, root = random_small_tree(rng, 40, 9)
    assert layer_widths(root) == widths_oracle(root), vals
print("Bài 5: OK", layer_widths(build_tree([1, 3, 2, 5, 3, None, 9])))

Bài 5: OK [1, 2, 4]


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "in cây vào bảng theo vị trí xác định":**

```python
class Solution:
    def printTree(self, root: Optional[TreeNode]) -> List[List[str]]:
        def height(node):
            if node is None:
                return -1                          # đếm CẠNH: lá cao 0
            return 1 + max(height(node.left), height(node.right))

        def dfs(node, r, c):
            if node is None:
                return
            res[r][c] = str(node.val)              # nhớ str()
            if node.left or node.right:            # có con => r < h
                step = 2 ** (h - r - 1)            # độ lệch giảm một nửa theo tầng
                dfs(node.left, r + 1, c - step)
                dfs(node.right, r + 1, c + step)

        h = height(root)
        m, n = h + 1, 2 ** (h + 1) - 1
        res = [[""] * n for _ in range(m)]         # mỗi hàng là list RIÊNG
        dfs(root, 0, (n - 1) // 2)
        return res
```

**Ghi nhớ một câu:** *"Đo chiều cao (đếm cạnh) → tạo bảng `(h+1) × (2^(h+1) − 1)` → đặt gốc ở giữa → con lệch `2^(h−r−1)`."*

**Checklist trước khi nộp:**

- [ ] `height(None) = -1` (đếm cạnh), `m = h + 1`, `n = 2 ** (h + 1) - 1`.
- [ ] Ma trận tạo bằng `[[""] * n for _ in range(m)]`, **không** dùng `* m`.
- [ ] Gốc ở cột `(n - 1) // 2` (chia nguyên).
- [ ] Độ lệch `2 ** (h - r - 1)` (có `- 1`), chỉ tính khi nút có con.
- [ ] Dùng `str(node.val)`; kiểm tra `node is None`, không kiểm tra giá trị `0`.
- [ ] Đã thử: một nút, cây lệch trái/phải, cây đầy, giá trị âm / `0` / hai chữ số, `h = 10`.
- [ ] Đã kiểm tra bằng hàm `check_layout` đi theo quy tắc của đề.